In [ ]:
# Cell 1
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedShuffleSplit, ParameterGrid
from sklearn.metrics import (
    precision_score,
    recall_score,
    roc_auc_score,
    accuracy_score,
    f1_score
)
from imblearn.over_sampling import RandomOverSampler

In [ ]:
# Cell 2
# Validation settings
n_splits = 5
seed = 10

kf = StratifiedShuffleSplit(
    n_splits=n_splits,
    test_size=1 / n_splits,
    random_state=seed
)

In [ ]:
# Cell 3
# Data preparation
def prepare_fold_data(X_train, X_val):
    X_train = X_train.copy()
    X_val = X_val.copy()

    # Convert boolean features to numeric
    bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
    bool_cols_val = X_val.select_dtypes(include=["bool"]).columns

    if len(bool_cols_train) > 0:
        X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)
    if len(bool_cols_val) > 0:
        X_val[bool_cols_val] = X_val[bool_cols_val].astype(int)

    # Dummy encoding and column alignment
    X_train = pd.get_dummies(X_train, drop_first=False)
    X_val = pd.get_dummies(X_val, drop_first=False)

    X_train, X_val = X_train.align(X_val, join="left", axis=1, fill_value=0)

    return X_train, X_val

In [ ]:
# Cell 4
# Evaluate one parameter set
def evaluate_rf_params(prefix, params, upsample=True):
    X = pd.read_csv(f"data/{prefix}_X_train.csv", keep_default_na=False)
    y = pd.read_csv(f"data/{prefix}_y_train.csv", keep_default_na=False).values.ravel()

    cv_scores = {
        "fold": [],
        "test_precision": [],
        "test_recall": [],
        "test_roc_auc": [],
        "test_accuracy": [],
        "test_f1": []
    }

    splits = list(kf.split(X, y))

    for fold_idx, (train_idx, val_idx) in enumerate(splits):
        X_train = X.iloc[train_idx].copy()
        y_train = y[train_idx].copy()

        X_val = X.iloc[val_idx].copy()
        y_val = y[val_idx].copy()

        # Oversampling is applied only to the training fold
        if upsample:
            upsampler = RandomOverSampler(random_state=seed)
            X_train, y_train = upsampler.fit_resample(X_train, y_train)

            if not isinstance(X_train, pd.DataFrame):
                X_train = pd.DataFrame(X_train, columns=X.columns)

        X_train_prepared, X_val_prepared = prepare_fold_data(X_train, X_val)

        clf = RandomForestClassifier(
            n_estimators=250,
            bootstrap=True,
            random_state=seed,
            n_jobs=-1,
            **params
        )

        clf.fit(X_train_prepared, y_train)

        y_pred = clf.predict(X_val_prepared)
        y_prob = clf.predict_proba(X_val_prepared)[:, 1]

        cv_scores["fold"].append(fold_idx + 1)
        cv_scores["test_precision"].append(precision_score(y_val, y_pred))
        cv_scores["test_recall"].append(recall_score(y_val, y_pred))
        cv_scores["test_roc_auc"].append(roc_auc_score(y_val, y_prob))
        cv_scores["test_accuracy"].append(accuracy_score(y_val, y_pred))
        cv_scores["test_f1"].append(f1_score(y_val, y_pred))

    # Convert fold metrics to arrays for aggregation
    for key in ["test_precision", "test_recall", "test_roc_auc", "test_accuracy", "test_f1"]:
        cv_scores[key] = np.array(cv_scores[key])

    summary = {
        "params": params,
        "precision_mean": cv_scores["test_precision"].mean(),
        "recall_mean": cv_scores["test_recall"].mean(),
        "roc_auc_mean": cv_scores["test_roc_auc"].mean(),
        "accuracy_mean": cv_scores["test_accuracy"].mean(),
        "f1_mean": cv_scores["test_f1"].mean()
    }

    return cv_scores, summary


In [ ]:
# Cell 5
# Hyperparameter grid
rf_param_grid = {
    "max_depth": [5, 10, 20],
    "criterion": ["gini", "entropy"],
    "min_samples_split": [2, 10, 20],
    "min_samples_leaf": [1, 5, 10],
    "max_features": ["sqrt", "log2", None]
}

In [ ]:
# Cell 6
# Grid search for best parameters
def tune_random_forest(prefix, upsample=True):
    all_results = []

    for params in ParameterGrid(rf_param_grid):
        _, summary = evaluate_rf_params(prefix, params, upsample=upsample)
        all_results.append(summary)
    
    results_df = pd.DataFrame(all_results)
    # Rank parameter settings by ROC AUC, then F1, then Accuracy
    results_df = results_df.sort_values(
        by=["roc_auc_mean", "f1_mean", "accuracy_mean"],
        ascending=False
    ).reset_index(drop=True)

    best_row = results_df.iloc[0]
    best_params = best_row["params"]

    print(f"Best params for {prefix}:")
    print(best_params)
    print("Best mean ROC AUC:", round(best_row["roc_auc_mean"], 3))
    print("Best mean precision:", round(best_row["precision_mean"], 3))
    print("Best mean recall:", round(best_row["recall_mean"], 3))
    print("Best mean accuracy:", round(best_row["accuracy_mean"], 3))
    print("Best mean F1:", round(best_row["f1_mean"], 3))

    return results_df, best_params

In [ ]:
# Cell 7
# Evaluate best model with CV + SE
def cross_validate_best_rf(prefix, best_params, upsample=True):
    cv_scores, _ = evaluate_rf_params(prefix, best_params, upsample=upsample)

    summary_df = pd.DataFrame({
        "fold": cv_scores["fold"],
        "precision": cv_scores["test_precision"],
        "recall": cv_scores["test_recall"],
        "roc_auc": cv_scores["test_roc_auc"],
        "accuracy": cv_scores["test_accuracy"],
        "f1": cv_scores["test_f1"]
    })

    mean_row = pd.DataFrame([{
        "fold": "mean",
        "precision": summary_df["precision"].mean(),
        "recall": summary_df["recall"].mean(),
        "roc_auc": summary_df["roc_auc"].mean(),
        "accuracy": summary_df["accuracy"].mean(),
        "f1": summary_df["f1"].mean()
    }])

    # Cross-validated standard error
    n = n_splits

    se_row = pd.DataFrame([{
        "fold": "se",
        "precision": summary_df["precision"].std(ddof=1) / np.sqrt(n),
        "recall": summary_df["recall"].std(ddof=1) / np.sqrt(n),
        "roc_auc": summary_df["roc_auc"].std(ddof=1) / np.sqrt(n),
        "accuracy": summary_df["accuracy"].std(ddof=1) / np.sqrt(n),
        "f1": summary_df["f1"].std(ddof=1) / np.sqrt(n)
    }])

    summary_df = pd.concat([summary_df, mean_row, se_row], ignore_index=True)

    print(f"Results for {prefix} (upsample={upsample})")

    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value = summary_df.loc[summary_df["fold"] == "mean", metric].values[0]
        se_value = summary_df.loc[summary_df["fold"] == "se", metric].values[0]
        print(f"Mean {metric}: {mean_value:.3f} ± {se_value:.3f}")

    return cv_scores, summary_df

In [ ]:
# Cell 8
# Run for all tasks
rf_results_mask_before, best_rf_params_mask_before = tune_random_forest(
    "mask_before",
    upsample=True
)
cv_scores_rf_mask_before, summary_rf_mask_before = cross_validate_best_rf(
    "mask_before",
    best_rf_params_mask_before,
    upsample=True
)

rf_results_mask_after, best_rf_params_mask_after = tune_random_forest(
    "mask_after",
    upsample=True
)
cv_scores_rf_mask_after, summary_rf_mask_after = cross_validate_best_rf(
    "mask_after",
    best_rf_params_mask_after,
    upsample=True
)

rf_results_general_before, best_rf_params_general_before = tune_random_forest(
    "general_before",
    upsample=True
)
cv_scores_rf_general_before, summary_rf_general_before = cross_validate_best_rf(
    "general_before",
    best_rf_params_general_before,
    upsample=True
)

rf_results_general_after, best_rf_params_general_after = tune_random_forest(
    "general_after",
    upsample=True
)
cv_scores_rf_general_after, summary_rf_general_after = cross_validate_best_rf(
    "general_after",
    best_rf_params_general_after,
    upsample=True
)

Best params for mask_before:
{'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2}
Best mean ROC AUC: 0.854
Best mean precision: 0.647
Best mean recall: 0.58
Best mean accuracy: 0.809
Best mean F1: 0.612
Results for mask_before (upsample=True)
Mean precision: 0.647 ± 0.007
Mean recall: 0.580 ± 0.005
Mean roc_auc: 0.854 ± 0.002
Mean accuracy: 0.809 ± 0.003
Mean f1: 0.612 ± 0.004
Best params for mask_after:
{'criterion': 'entropy', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2}
Best mean ROC AUC: 0.886
Best mean precision: 0.875
Best mean recall: 0.911
Best mean accuracy: 0.845
Best mean F1: 0.892
Results for mask_after (upsample=True)
Mean precision: 0.875 ± 0.001
Mean recall: 0.911 ± 0.004
Mean roc_auc: 0.886 ± 0.002
Mean accuracy: 0.845 ± 0.003
Mean f1: 0.892 ± 0.002
Best params for general_before:
{'criterion': 'entropy', 'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 1, 

In [ ]:
# Cell 9
# Build comparison table
def get_mean_se(summary_df, metric):
    mean_value = summary_df.loc[summary_df["fold"] == "mean", metric].values[0]
    se_value = summary_df.loc[summary_df["fold"] == "se", metric].values[0]
    return mean_value, se_value


summaries = {
    "mask_before": summary_rf_mask_before,
    "mask_after": summary_rf_mask_after,
    "general_before": summary_rf_general_before,
    "general_after": summary_rf_general_after
}

rows = []

for model_name, summary in summaries.items():
    row = {"model": model_name}
    
    for metric in ["precision", "recall", "roc_auc", "accuracy", "f1"]:
        mean_value, se_value = get_mean_se(summary, metric)
        row[metric] = mean_value
        row[f"{metric}_se"] = se_value
        row[f"{metric}_mean_se"] = f"{mean_value:.3f} ± {se_value:.3f}"
    
    rows.append(row)

comparison_df = pd.DataFrame(rows)

comparison_df

,model,precision,precision_se,precision_mean_se,recall,recall_se,recall_mean_se,roc_auc,roc_auc_se,roc_auc_mean_se,accuracy,accuracy_se,accuracy_mean_se,f1,f1_se,f1_mean_se
0,mask_before,0.647256,0.007320,0.647 ± 0.007,0.580286,0.005368,0.580 ± 0.005,0.854048,0.002425,0.854 ± 0.002,0.809068,0.002723,0.809 ± 0.003,0.611818,0.004482,0.612 ± 0.004
1,mask_after,0.874854,0.000506,0.875 ± 0.001,0.910746,0.003864,0.911 ± 0.004,0.886346,0.001554,0.886 ± 0.002,0.844829,0.002596,0.845 ± 0.003,0.892425,0.002013,0.892 ± 0.002
2,general_before,0.726081,0.003768,0.726 ± 0.004,0.744790,0.003357,0.745 ± 0.003,0.788916,0.004814,0.789 ± 0.005,0.715746,0.003621,0.716 ± 0.004,0.735304,0.003225,0.735 ± 0.003
3,general_after,0.845550,0.001826,0.846 ± 0.002,0.882485,0.001521,0.882 ± 0.002,0.839208,0.000918,0.839 ± 0.001,0.798021,0.001922,0.798 ± 0.002,0.863617,0.001237,0.864 ± 0.001


In [ ]:
# Cell 10
pd.DataFrame(comparison_df).to_csv("results/rf_mask_before.csv", index=False)